In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from category_encoders import TargetEncoder
import statsmodels.api as sm
from sklearn.model_selection import learning_curve
from sklearn.metrics import roc_curve, precision_recall_curve, auc, precision_score, recall_score, f1_score, roc_auc_score
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import shap
from PyALE import ale
from sklearn.neural_network import MLPClassifier
import tensorflow as tf
import keras

from sklearn import set_config
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

Initialize

In [2]:
df = pd.read_csv('..\Data\loan_data_sample.csv')
features = ['loan_amnt', 'term', 'emp_length', 'home_ownership',
       'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'initial_list_status', 'mths_since_last_major_derog',
       'application_type', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
       'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc',
       'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'chargeoff_within_12_mths', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
       'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq',
       'mths_since_recent_inq', 'mths_since_recent_revol_delinq',
       'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
       'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
       'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
       'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit',
       'total_il_high_credit_limit',
       'months_sincefrst_credit', 'public_record', 'is_consolidation',
       'addr_state', 'is_currently_delinq', 'has_il_history']

index_sql = 'Loan_ID'
target = 'predictor'

df_features  = df[features]
df_predictor = pd.Series(df[target])


#Imputing col's we imputed with 999 in SQL
imputed_cols = [
    'mths_since_last_delinq', 'mths_since_last_record', 
    'mths_since_last_major_derog', 'mths_since_recent_bc_dlq', 
    'mths_since_recent_inq', 'mths_since_recent_revol_delinq'
]

df_features.loc[:,imputed_cols] = df_features[imputed_cols].replace(999.0, np.nan)





X_train, X_test, y_train,y_test = train_test_split(df_features,df_predictor,stratify=df_predictor,test_size=.2,random_state=11)

categorical_features = X_train.select_dtypes(include=['object','category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()

Pipeline for Flags / missing values  --- Target Encoding

In [3]:
zero_cols = [
    'max_bal_bc', 'all_util', 'il_util', 'open_acc_6m', 
    'open_il_12m', 'open_il_24m', 'open_rv_12m', 'open_rv_24m', 'inq_last_12m',
    'open_act_il', 'total_bal_il', 'total_il_high_credit_limit', 'is_consolidation'
]
flag_cols = [
    'mths_since_last_delinq', 'mths_since_last_record', 
    'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 
    'mths_since_recent_inq', 'mths_since_rcnt_il',
    'mths_since_last_major_derog'
]
median_cols = [
    'months_sincefrst_credit', 'annual_inc', 'inq_last_6mths', 
    'revol_util', 'total_acc', 'pub_rec', 'open_acc', 
    'mo_sin_old_rev_tl_op', 'num_rev_accts', 'tot_hi_cred_lim',
    'acc_open_past_24mths', 'num_bc_sats', 'num_sats', 'mort_acc',
    'mths_since_recent_bc', 'total_bc_limit', 'pub_rec_bankruptcies',
    'total_rev_hi_lim', 'inq_fi', 'avg_cur_bal', 'bc_open_to_buy', 
    'bc_util', 'mo_sin_old_il_acct', 'mo_sin_rcnt_rev_tl_op', 
    'mo_sin_rcnt_tl', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 
    'num_actv_rev_tl', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 
    'num_rev_tl_bal_gt_0', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 
    'pct_tl_nvr_dlq', 'percent_bc_gt_75',
    'total_cu_tl', 'total_bal_ex_mort', 'num_tl_30dpd', 'num_tl_120dpd_2m', 'chargeoff_within_12_mths'
]

ohe_cols = ['home_ownership', 'verification_status', 'application_type'] 
target_col = ['addr_state']

def make_ratio(X):
    eps = 0.001
    return ( X[:, [0]] / (X[:, [1]] + eps) )

def monthlycash(X):
    return ((X[:, [0]] / 12) * (1 - (X[:, [1]] / 100)))

def ratio_name(function_transformer, feature_names_in):
    return ['custom_ratio'] 

#Winsorizer 
from scipy.stats.mstats import winsorize

def winsorize_fn(X):
    return np.array(winsorize(np.array(X), limits=[0.01, 0.01], axis=0))

def make_winsorizer():
    return FunctionTransformer(winsorize_fn,feature_names_out='one-to-one')

zero_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value=0)),
    ('winsorize', make_winsorizer()),
    ('scale', StandardScaler())
])

median_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('winsorize', make_winsorizer()),
    ('scale', StandardScaler())
])

flag_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True)),
])

cat_ohe_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)) 
])

target_enc_pipe = Pipeline([
    ('target_encode', TargetEncoder(smoothing=10, handle_missing='value', handle_unknown='value')),
    ('scale', StandardScaler())
])

def ratio_pipe():
    return Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('ratio', FunctionTransformer(make_ratio, feature_names_out=ratio_name)),
        ('winsorize', make_winsorizer()),
        ('scale', StandardScaler())])

monthly_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('calc', FunctionTransformer(monthlycash, feature_names_out=ratio_name)),
    ('winsorize', make_winsorizer()),
    ('scale', StandardScaler())
])

set_config(transform_output="default")
preprocessor = ColumnTransformer(
    transformers=[
        ('zeros', zero_pipe, zero_cols),
        ('medians', median_pipe, median_cols),
        ('flags', flag_pipe, flag_cols),
        ('ohe', cat_ohe_pipe, ohe_cols),
        ('target', target_enc_pipe, target_col),
        
        #these are our ratio's
        ('FE_loan_to_income', ratio_pipe(), ['loan_amnt', 'annual_inc']),
        ('FE_activity_ratio', ratio_pipe(), ['num_actv_rev_tl', 'num_op_rev_tl']),
        ('my_monthly_cash', monthly_pipe, ['annual_inc', 'dti']) 
    ],
    remainder='drop' 
)



In [4]:
X_train_pre = preprocessor.fit_transform(X_train, y_train)
X_test_pre = preprocessor.transform(X_test)

# convert to pandas
X_train_pre = pd.DataFrame(X_train_pre, columns=preprocessor.get_feature_names_out())
X_test_pre = pd.DataFrame(X_test_pre, columns=preprocessor.get_feature_names_out())

In [6]:
print(len(X_train_pre.columns),len(X_train.columns))

print(X_train_pre.shape)

79 79
(104511, 79)


In [12]:
X_train1 , X_val, y_train1, y_val = train_test_split(X_train_pre, y_train, stratify=y_train, train_size = 0.8,
                                                      random_state=11)


model = tf.keras.Sequential ([
    tf.keras.layers.Dense(100,activation ='relu',input_shape=(X_train_pre.shape[1],)),
    tf.keras.layers.Dense(50,activation='relu'),
    tf.keras.layers.Dense(1,activation='sigmoid')
])
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['AUC'])

tensorboard_cb = tf.keras.callbacks.TensorBoard(log_dir='./baseline')
early_stop_cb = tf.keras.callbacks.EarlyStopping(patience=20,restore_best_weights=True)


history = model.fit(
    X_train1,y_train1,
    validation_data=(X_val,y_val),
    epochs=100,
    callbacks=[tensorboard_cb],
    batch_size=256
)

c:\Users\Marwa\anaconda3\envs\data_cleaning\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
327/327 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - AUC: 0.6320 - loss: 0.5022 - val_AUC: 0.6746 - val_loss: 0.4808
Epoch 2/100
327/327 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - AUC: 0.6654 - loss: 0.4757 - val_AUC: 0.6798 - val_loss: 0.4692
Epoch 3/100
327/327 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - AUC: 0.6748 - loss: 0.4710 - val_AUC: 0.6811 - val_loss: 0.4792
Epoch 4/100
327/327 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - AUC: 0.6699 - loss: 0.4742 - val_AUC: 0.6841 - val_loss: 0.4669
Epoch 5/100
327/327 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - AUC: 0.6779 - loss: 0.4700 - val_AUC: 0.6845 - val_loss: 0.4668
Epoch 6/100
327/327 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - AUC: 0.6816 - loss: 0.4680 - val_AUC: 0.6826 - val_loss: 0.4717
Epoch 7/100
327/327 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - AUC: 0.6790 - loss: 0.4694 - val_AUC: 0.6766 - val_loss: 0.4795
Epoch 8/100
327/327 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - AUC: 0.6849 - loss: 0.4663 - val_AUC: 0.6855 - val_loss: 0.4686
Epoch 9/100
327/327 ━━━━━━━━━━━━━━━━━━━━ 2s 5m